The goal of this notebook is to make the figures of the SE BGC poster from the `bgc_20241001_20250701` experiment (Gulf Stream region, 2024-10-01 to 2025-07-01). Each section below matches one panel group of the poster.

- Study region and data: absolute dynamic topography with a locator inset, and the domain-mean PACE Rrs spectrum on 2025-03-12.
- Eddy identification and tracking: every cyclone and anticyclone track, and the contours of cyclone 13 and anticyclone 3 every 10 days.
- PACE collocation: Rrs(442) inside cyclone 13 on 2025-03-29.
- SDP: the diagnostic pigments of cyclone 13 on 2025-01-28.
- Results: mean total chlorophyll-a over time by polarity.

Set `SAVE_DIR` to a folder to also write the 300 dpi PNGs.

In [ ]:
from calendar import monthrange
from datetime import datetime
from pathlib import Path
from typing import cast

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import xarray as xr
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.collections import LineCollection
from matplotlib.figure import Figure
from matplotlib.path import Path as MplPath
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

from eddy_tracking.config import load_config, resolve_data_dir, resolve_output_dir
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations

EXPERIMENT = "bgc_20241001_20250701"
SAVE_DIR: Path | None = None  # set to a folder to also write 300 dpi PNGs for the poster
LON_RANGE = [-81, -56]
LAT_RANGE = [29, 44]
TRACK_ID = 13  # the cyclone shown in the PACE collocation and SDP panels
cfg = load_config(EXPERIMENT)
proj = ccrs.PlateCarree()

## Study Region & Data

Region: Northwest Atlantic Gulf Stream

Lon: -81 to -56

Lat: 29 to 44

Date Snapshot Used: 2025-03-12

In [ ]:
SNAPSHOT = "20250312"

ssh = xr.open_dataset(next(resolve_data_dir(cfg, "swot_dir").glob(f"dt_global_allsat_phy_l4_{SNAPSHOT}_*.nc")))
# (1, n_lat, n_lon) -> (n_lat, n_lon)
adt = ssh["adt"].squeeze()

pace = xr.open_dataset(next(resolve_data_dir(cfg, "pace_dir").glob(f"PACE_OCI.{SNAPSHOT}.*.4km.nc")))
wl = pace["wavelength"].values
# (n_lat, n_lon, n_wl) -> (n_wl,)
rrs_domain_mean = np.nanmean(pace["Rrs"].values, axis=(0, 1))


In [ ]:
fig = cast(Figure, plt.figure(figsize=(10, 8.5), dpi=300))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.25)
ax1 = cast(GeoAxes, fig.add_subplot(gs[0], projection=proj))
ax2 = fig.add_subplot(gs[1])

ax1.set_extent([*LON_RANGE, *LAT_RANGE], crs=proj)
vlim = np.ceil(max(abs(float(adt.min())), abs(float(adt.max()))) * 4) / 4
cf = ax1.contourf(
    adt.longitude, adt.latitude, adt.values,
    levels=np.linspace(-vlim, vlim, 40), cmap="RdBu_r",
    transform=proj, zorder=0, extend="both",
)
ax1.add_feature(cfeature.NaturalEarthFeature("physical", "land", "10m",
        facecolor="#e8e8e8", edgecolor="none"), zorder=1)
ax1.add_feature(cfeature.NaturalEarthFeature("physical", "coastline", "10m",
        facecolor="none", edgecolor="0.3", linewidth=0.6), zorder=2)
for name, lon, lat, sz, rot in [("Gulf Stream", -70, 38.5, 13, 0),
                                 ("Sargasso Sea", -62, 30.5, 12, 0)]:
    ax1.text(lon, lat, name, fontsize=sz, fontstyle="italic", color="0.15",
        ha="center", va="center", rotation=rot, transform=proj, zorder=5)
gl1 = ax1.gridlines(draw_labels=True, linewidth=0.3, color="0.5", alpha=0.5, zorder=4)
gl1.top_labels = gl1.right_labels = False
gl1.xlabel_style = {"size": 11}
gl1.ylabel_style = {"size": 11}
cbar1 = fig.colorbar(cf, ax=ax1, orientation="vertical", shrink=0.75, pad=0.02, aspect=25)
cbar1.set_ticks(np.arange(-vlim, vlim + 0.125, 0.5))
cbar1.set_label("(m)", fontsize=11)
cbar1.ax.tick_params(labelsize=10)
ax1.set_title("(a) Absolute Dynamic Topography", fontsize=13, pad=10)

inset_ax = inset_axes(ax1, width="22%", height="30%", loc="upper left",
    axes_class=GeoAxes,
    axes_kwargs={"projection": proj}, borderpad=1.5)
inset_ax.set_extent([-100, -30, 10, 55], crs=proj)
inset_ax.add_feature(cfeature.LAND, facecolor="#e8e8e8", edgecolor="0.5", linewidth=0.3)
inset_ax.add_feature(cfeature.OCEAN, facecolor="#dbe9f6")
inset_ax.add_patch(mpatches.Rectangle(
    (LON_RANGE[0], LAT_RANGE[0]),
    LON_RANGE[1] - LON_RANGE[0], LAT_RANGE[1] - LAT_RANGE[0],
    linewidth=1.5, edgecolor="red", facecolor="red", alpha=0.25, transform=proj,
))

ax2.plot(wl, rrs_domain_mean, color="#006d6f", lw=2)
ax2.set_xlabel("Wavelength (nm)", fontsize=12)
ax2.set_ylabel(r"$R_{\mathrm{rs}}$ (sr$^{-1}$)", fontsize=12)
ax2.set_xlim(345, 720)
ax2.set_ylim(bottom=0)
ax2.tick_params(labelsize=11)
ax2.set_title(r"(b) Remote Sensing Reflectance ($R_{\mathrm{rs}}$)", fontsize=13, pad=10)
ax2.spines[["top", "right"]].set_visible(False)

# get_position() returns the pre-layout box until a draw runs.
fig.canvas.draw()
pos1 = ax1.get_position()
cbar_right = cbar1.ax.get_position().x1
pos2 = ax2.get_position()
ax2.set_position([pos1.x0, pos2.y0, cbar_right - pos1.x0, pos1.height])

pos2_new = ax2.get_position()
mid_y = (pos1.y0 + pos2_new.y1) / 2
fig.add_artist(plt.Line2D([pos1.x0, cbar_right], [mid_y, mid_y],
        transform=fig.transFigure, color="0.75", linewidth=0.8))

if SAVE_DIR is not None:
    fig.savefig(SAVE_DIR / "study_region_map.png", dpi=300, bbox_inches="tight")
plt.show()

## Eddy Identification & Tracking

In [ ]:
HIGHLIGHT = {"cyclone": 13, "anticyclone": 3}

tracks = {}
for pol in ("cyclone", "anticyclone"):
    path = resolve_output_dir(EXPERIMENT, "eddy_track", pol) / f"{pol}_tracks.zarr"
    t = TrackEddiesObservations.load_file(str(path))
    lons = t["longitude"].copy()
    lons[lons > 180] -= 360
    tracks[pol] = {"obs": t, "lon": lons, "lat": t["latitude"], "ids": np.unique(t["track"])}

In [ ]:
COLORS = {"cyclone": "#2166ac", "anticyclone": "#b2182b"}
LABEL_POS_A = {"cyclone": (-60, 39.0), "anticyclone": (-71, 34.4)}
LABEL_POS_B = {"cyclone": (-60, 39.0), "anticyclone": (-71, 34.4)}

def draw_gradient_track(ax, lons, lats, color, lw=2.5, alpha_range=(0.2, 0.95), zorder=6):
    # (2, n_obs) -> (n_obs, 2) -> (n_obs, 1, 2)
    points = np.array([lons, lats]).T.reshape(-1, 1, 2)
    # two (n_obs-1, 1, 2) -> (n_obs-1, 2, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    n = len(segments)
    alphas = np.linspace(alpha_range[0], alpha_range[1], n)
    rgba = mcolors.to_rgba(color)
    colors = [(rgba[0], rgba[1], rgba[2], a) for a in alphas]
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=lw, transform=proj, zorder=zorder))

def draw_id_label(ax, lon, lat, text, color):
    ax.text(lon, lat, text, fontsize=8, fontweight="bold", color=color,
        ha="center", va="center", transform=proj, zorder=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=color, alpha=0.92, linewidth=0.8))

fig = cast(Figure, plt.figure(figsize=(10, 7), dpi=300))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.35)
ax1 = cast(GeoAxes, fig.add_subplot(gs[0], projection=proj))
ax2 = cast(GeoAxes, fig.add_subplot(gs[1], projection=proj))

for ax in (ax1, ax2):
    ax.set_extent([*LON_RANGE, *LAT_RANGE], crs=proj)
    ax.add_feature(cfeature.NaturalEarthFeature("physical", "land", "10m",
            facecolor="#e8e8e8", edgecolor="none"), zorder=1)
    ax.add_feature(cfeature.NaturalEarthFeature("physical", "coastline", "10m",
            facecolor="none", edgecolor="0.3", linewidth=0.6), zorder=2)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.5", alpha=0.5, zorder=4)
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}

for pol in ("cyclone", "anticyclone"):
    t = tracks[pol]
    hid = HIGHLIGHT[pol]
    color = COLORS[pol]
    for tid in t["ids"]:
        mask = t["obs"]["track"] == tid
        lx, ly = t["lon"][mask], t["lat"][mask]
        if tid != hid:
            draw_gradient_track(ax1, lx, ly, color, lw=0.8, alpha_range=(0.12, 0.3), zorder=3)
            ax1.plot(lx[0], ly[0], "o", color=color, ms=2.5, mfc="white", mew=0.5,
                alpha=0.4, transform=proj, zorder=4)
            ax1.plot(lx[-1], ly[-1], "o", color=color, ms=2.5, mfc=color, mew=0.5,
                alpha=0.4, transform=proj, zorder=4)
        else:
            draw_gradient_track(ax1, lx, ly, color, lw=2.0, alpha_range=(0.3, 0.95), zorder=6)
            ax1.plot(lx[0], ly[0], "o", color=color, ms=5, mfc="white", mew=1.0,
                transform=proj, zorder=8)
            ax1.plot(lx[-1], ly[-1], "o", color=color, ms=5, mfc=color, mew=1.0,
                transform=proj, zorder=8)
    lon_lbl, lat_lbl = LABEL_POS_A[pol]
    draw_id_label(ax1, lon_lbl, lat_lbl, f"#{hid}", color)

n_cyc = len(tracks["cyclone"]["ids"])
n_anti = len(tracks["anticyclone"]["ids"])
ax1.legend(handles=[
    plt.Line2D([], [], color=COLORS["cyclone"], lw=2.5, label=f"Cyclone ({n_cyc})"),
    plt.Line2D([], [], color=COLORS["anticyclone"], lw=2.5, label=f"Anticyclone ({n_anti})"),
], loc="upper right", fontsize=7, frameon=True, facecolor="white",
   edgecolor="0.8", framealpha=0.95, handlelength=1.5)
ax1.set_title("(a) Eddy Tracks", fontsize=11, pad=8)

for pol in ("cyclone", "anticyclone"):
    t_data = tracks[pol]["obs"]
    hid = HIGHLIGHT[pol]
    color = COLORS[pol]
    mask = t_data["track"] == hid
    indices = np.where(mask)[0]
    lx = tracks[pol]["lon"][mask]
    ly = tracks[pol]["lat"][mask]
    n_obs = len(indices)

    draw_gradient_track(ax2, lx, ly, color, lw=1.2, alpha_range=(0.15, 0.5))

    contour_idx = list(range(0, n_obs, 10))
    if (n_obs - 1) not in contour_idx:
        contour_idx.append(n_obs - 1)
    n_c = len(contour_idx)

    for j, ci in enumerate(contour_idx):
        idx = indices[ci]
        clons = t_data["contour_lon_e"][idx].copy()
        clats = t_data["contour_lat_e"][idx].copy()
        clons[clons > 180] -= 360
        # (n_pts,) -> (n_pts + 1,): repeat the first vertex to close the ring
        clons = np.append(clons, clons[0])
        clats = np.append(clats, clats[0])
        alpha = 0.15 + 0.7 * (j / max(n_c - 1, 1))
        ax2.plot(clons, clats, color=color, lw=0.8, alpha=alpha, transform=proj, zorder=5)
        ax2.fill(clons, clats, color=color, alpha=alpha * 0.1, transform=proj, zorder=4)
        ax2.plot(lx[ci], ly[ci], ".", color=color, ms=4, alpha=alpha, transform=proj, zorder=6)

    lon_lbl, lat_lbl = LABEL_POS_B[pol]
    draw_id_label(ax2, lon_lbl, lat_lbl, f"#{hid}", color)

ax2.set_title("(b) Eddy Contours (10-day intervals)", fontsize=11, pad=8)

# get_position() returns the pre-layout box until a draw runs.
fig.canvas.draw()
pos1 = ax1.get_position()
pos2 = ax2.get_position()
ax2.set_position([pos1.x0, pos2.y0, pos1.width, pos1.height])

pos2_new = ax2.get_position()
mid_y = (pos1.y0 + pos2_new.y1) / 2
fig.add_artist(plt.Line2D([pos1.x0, pos1.x0 + pos1.width], [mid_y, mid_y],
        transform=fig.transFigure, color="0.75", linewidth=0.8))

if SAVE_DIR is not None:
    fig.savefig(SAVE_DIR / "eddy_id_and_tracking.png", dpi=300, bbox_inches="tight")
plt.show()

## PACE Collocation

In [ ]:
COLLOCATION_DATE = "2025-03-29"
WAVELENGTH = 442

date_str = COLLOCATION_DATE.replace("-", "")
pace_path = next(resolve_data_dir(load_config(EXPERIMENT), "pace_dir").glob(f"PACE_OCI.{date_str}.*.4km.nc"))
pace = xr.open_dataset(pace_path)

wl = pace["wavelength"].values
wl_idx = np.argmin(np.abs(wl - WAVELENGTH))

tracks_path = resolve_output_dir(EXPERIMENT, "eddy_track", "cyclone") / "cyclone_tracks.zarr"
t = TrackEddiesObservations.load_file(str(tracks_path))

mask = t["track"] == TRACK_ID
obs_idx = np.where(mask)[0]
epoch = np.datetime64("1950-01-01")
target_day = (np.datetime64(COLLOCATION_DATE) - epoch) / np.timedelta64(1, "D")
closest = np.argmin(np.abs(t["time"][mask] - target_day))
idx = obs_idx[closest]

clons = t["contour_lon_e"][idx].copy()
clats = t["contour_lat_e"][idx].copy()
clons[clons > 180] -= 360
# (n_pts,) -> (n_pts + 1,): repeat the first vertex to close the ring
poly_lons = np.append(clons, clons[0])
poly_lats = np.append(clats, clats[0])

print(
    f"eddy_track_id: {TRACK_ID}\n"
    f"date: {COLLOCATION_DATE}"
)
print(f"effective_radius_km: {t['radius_e'][idx]/1000:.1f}")

In [ ]:
# Square symmetric extent snapped to 0.5° gridline alignment
center_lon = (clons.min() + clons.max()) / 2
center_lat = (clats.min() + clats.max()) / 2
half_span = max(clons.max() - clons.min(), clats.max() - clats.min()) / 2 + 0.7

def round_half(x, direction):
    return np.floor(x * 2) / 2 if direction == "down" else np.ceil(x * 2) / 2

west = round_half(center_lon - half_span, "down")
east = round_half(center_lon + half_span, "up")
south = round_half(center_lat - half_span, "down")
north = round_half(center_lat + half_span, "up")
w, h = east - west, north - south
if w < h:
    diff = h - w
    west = round_half(west - diff/2, "down")
    east = round_half(east + diff/2, "up")
elif h < w:
    diff = w - h
    south = round_half(south - diff/2, "down")
    north = round_half(north + diff/2, "up")

# Trim latitude padding to make the plot less tall
LAT_INSET = 0.5
south += LAT_INSET
north -= LAT_INSET
extent = [west, east, south, north]

# Subset PACE grid
lon = pace["lon"].values
lat = pace["lat"].values
lon_mask = (lon >= extent[0]) & (lon <= extent[1])
lat_mask = (lat >= extent[2]) & (lat <= extent[3])
lon_sub = lon[lon_mask]
lat_sub = lat[lat_mask]
# (n_lat, n_lon, n_wl) -> (n_lat_sub, n_lon_sub, n_wl) -> (n_lat_sub, n_lon_sub)
rrs_sub = pace["Rrs"].values[np.ix_(lat_mask, lon_mask)][:, :, wl_idx]

# two (n_pts,) -> (n_pts, 2)
contour_path = MplPath(np.column_stack([clons, clats]))
# (n_lon_sub,), (n_lat_sub,) -> two (n_lat_sub, n_lon_sub)
lon_grid, lat_grid = np.meshgrid(lon_sub, lat_sub)
# two (n_lat_sub, n_lon_sub) -> (n_lat_sub * n_lon_sub,) -> (n_lat_sub * n_lon_sub, 2)
points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])
# (n_lat_sub * n_lon_sub,) -> (n_lat_sub, n_lon_sub)
inside = contour_path.contains_points(points).reshape(lon_grid.shape)

n_inside = inside.sum()
n_valid = (inside & np.isfinite(rrs_sub)).sum()
coverage = n_valid / n_inside
print(f"pixels_inside_contour: {n_inside}")
print(f"valid_non_nan_pixels: {n_valid}")
print(f"within_eddy_coverage: {coverage:.3f}")

In [ ]:
fig, ax = cast("tuple[Figure, GeoAxes]", plt.subplots(figsize=(6.5, 4.8), dpi=300, subplot_kw={"projection": proj}))
ax.set_extent(extent, crs=proj)
ax.set_facecolor("white")

rrs_outside = rrs_sub.copy()
rrs_outside[inside] = np.nan
out_vmin = np.nanpercentile(rrs_outside, 5)
out_vmax = np.nanpercentile(rrs_outside, 95)
ax.pcolormesh(lon_sub, lat_sub, rrs_outside, cmap="Greys",
    vmin=out_vmin, vmax=out_vmax, transform=proj, zorder=1, alpha=0.4)

rrs_color = rrs_sub.copy()
rrs_color[~inside] = np.nan
vmin = np.nanpercentile(rrs_sub[inside], 2)
vmax = np.nanpercentile(rrs_sub[inside], 98)
cf = ax.pcolormesh(lon_sub, lat_sub, rrs_color, cmap="viridis",
    vmin=vmin, vmax=vmax, transform=proj, zorder=3)

ax.plot(poly_lons, poly_lats, color="#2166ac", lw=2.2, transform=proj, zorder=5)

ax.text(0.97, 0.97, f"Within-Eddy Coverage = {coverage:.2f}", transform=ax.transAxes,
    fontsize=10, ha="right", va="top",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.7", alpha=0.9),
    zorder=10)

gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.5", alpha=0.4, zorder=7)
gl.top_labels = gl.right_labels = False
gl.xlabel_style = {"size": 9}
gl.ylabel_style = {"size": 9}
gl.xlocator = mticker.FixedLocator(np.arange(extent[0], extent[1] + 0.1, 0.5))
gl.ylocator = mticker.FixedLocator(np.arange(extent[2], extent[3] + 0.1, 0.5))

cbar = fig.colorbar(cf, ax=ax, orientation="vertical", shrink=0.75, pad=0.06, aspect=25)
cbar.set_label(r"$R_{\mathrm{rs}}$(442) (sr$^{-1}$)", fontsize=10)
cbar.ax.tick_params(labelsize=9)

date_obj = datetime.strptime(COLLOCATION_DATE, "%Y-%m-%d")
ax.set_title(f"PACE Collocation: Cyclonic Eddy #{TRACK_ID} On {date_obj.strftime('%B %d, %Y')}",
    fontsize=11, pad=8)

if SAVE_DIR is not None:
    fig.savefig(SAVE_DIR / "pace_collocation.png", dpi=300, bbox_inches="tight")
plt.show()

## SDP: eddy-wide pigment summary

In [ ]:
PIGMENT_DATE = pd.Timestamp("2025-01-28")

# Diagnostic biomarker pigments (T chla shown separately as annotation since it dominates the scale)
PIGMENT_NAMES = ["DV chla", "MV chlb", "chl c1+c2", "chl c3",
    "Fuco", "ButFuco", "HexFuco", "Perid", "Allo", "Zea", "Viola", "Neo"]

PIGMENT_DISPLAY = [
    "DV Chl-a", "MV Chl-b", "Chl-c1+c2", "Chl-c3",
    "Fucoxanthin", "But-Fuco", "Hex-Fuco", "Peridinin",
    "Alloxanthin", "Zeaxanthin", "Violaxanthin", "Neoxanthin",
]

# Each biomarker pigment is colored by its primary PFT association
PIGMENT_COLORS = [
    "#1565C0",  # DV chla -> Cyanobacteria (Prochlorococcus)
    "#2E7D32",  # MV chlb -> Green algae
    "#FFB347",  # chl c1+c2 -> Haptophytes
    "#FFB347",  # chl c3 -> Haptophytes
    "#E87A14",  # Fuco -> Diatoms
    "#FFB347",  # ButFuco -> Haptophytes
    "#FFB347",  # HexFuco -> Haptophytes
    "#C62828",  # Perid -> Dinoflagellates
    "#AD1457",  # Allo -> Cryptophytes
    "#1565C0",  # Zea -> Cyanobacteria
    "#2E7D32",  # Viola -> Green algae
    "#2E7D32",  # Neo -> Green algae
]

pig = pd.read_parquet(resolve_output_dir(EXPERIMENT, "pigments", "cyclone") / f"eddy_{TRACK_ID}_pigments.parquet")

pig_d = pig[pig["date"] == PIGMENT_DATE].reset_index(drop=True)
n_pixels = len(pig_d)

pig_mean = pig_d[PIGMENT_NAMES].mean().to_numpy()  # pyright: ignore[reportAttributeAccessIssue]
pig_std = pig_d[PIGMENT_NAMES].std().to_numpy()  # pyright: ignore[reportAttributeAccessIssue]
total_chla_mean = pig_d["T chla"].mean()

print(f"pixels: {n_pixels}")
print(f"mean_total_chla_mg_m3: {total_chla_mean:.3f}")


In [ ]:
fig = cast(Figure, plt.figure(figsize=(7, 3.8), dpi=300))
ax1 = fig.add_subplot(111)

x_pos = np.arange(len(PIGMENT_NAMES))
ax1.bar(x_pos, pig_mean, yerr=pig_std, color=PIGMENT_COLORS,
    edgecolor="0.25", linewidth=0.5,
    error_kw=dict(ecolor="0.3", lw=0.8, capsize=2.5, capthick=0.8))
ax1.set_xticks(x_pos)
ax1.set_xticklabels(PIGMENT_DISPLAY, rotation=45, ha="right", fontsize=8)
ax1.set_ylabel(r"Concentration (mg m$^{-3}$)", fontsize=9)
ax1.tick_params(labelsize=8)
ax1.spines[["top", "right"]].set_visible(False)
ax1.set_title(f"Cyclone {TRACK_ID}: diagnostic pigments on {PIGMENT_DATE:%Y-%m-%d}", fontsize=10, pad=6)
ax1.text(0.98, 0.95, f"Total Chl-$a$ = {total_chla_mean:.2f} mg m$^{{-3}}$",
    transform=ax1.transAxes, fontsize=8, ha="right", va="top", color="0.3",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.7", linewidth=0.5))

if SAVE_DIR is not None:
    fig.savefig(SAVE_DIR / "sdp.png", dpi=300, bbox_inches="tight")
plt.show()

## Results

Mean chlorophyll-a concentration over time by polarity from the SDP pigment results.

In [ ]:
pig_frames = []
for polarity in ("cyclone", "anticyclone"):
    pig_dir = resolve_output_dir(EXPERIMENT, "pigments", polarity)
    for fp in sorted(pig_dir.glob("eddy_*_pigments.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"] = polarity
        pig_frames.append(df)
pigments = pd.concat(pig_frames, ignore_index=True)

pig_medians = (
    pigments.groupby(["track_id", "date", "polarity"])["T chla"]
    .mean().reset_index()
)
pig_medians = pig_medians[pig_medians["T chla"] >= 0.01].copy()

print(f"pigment_eddy_dates_t_chla_gte_0_01: {len(pig_medians):,}")


### Plot: Mean Chl-a Over Time by Polarity

Per-eddy-halfmonth means aggregated across eddies (one mean per polarity per half-month bin) with 95% bootstrap CIs (2000 resamples). Colored dots: individual eddy-date spatial means, positioned by actual observation date with low alpha so dense regions cumulatively darken. The single anticyclone June outlier (~2.2 mg/m³) is filtered.

In [ ]:
pig_m = pig_medians.copy()
pig_m["month"] = pig_m["date"].dt.month  # pyright: ignore[reportAttributeAccessIssue]
pig_m["half"] = (pig_m["date"].dt.day > 15).astype(int)  # pyright: ignore[reportAttributeAccessIssue]

# Per-eddy-halfmonth means for the line + CI computation
eddy_hm = (
    pig_m.groupby(["track_id", "polarity", "month", "half"])["T chla"]
    .mean().reset_index()
)

# Remove the anticyclone June outlier (single ~2.2 mg/m^3 point)
eddy_hm = eddy_hm[
    ~((eddy_hm["polarity"] == "anticyclone") & (eddy_hm["T chla"] > 1.5))
].reset_index(drop=True)

MONTH_ORDER = [10, 11, 12, 1, 2, 3, 4, 5, 6]
MONTH_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun"]

bins_sequence = [(m, h) for m in MONTH_ORDER for h in (0, 1)]
month_tick_positions = [i * 2 + 0.5 for i in range(len(MONTH_ORDER))]


def map_date_to_x(ts):
    """
    Map a date to a continuous x position on the half-month axis. Each bin occupies [center - 0.5, center + 0.5]; the fractional offset within the bin reflects the actual day of month.
    """
    if ts.month not in MONTH_ORDER:
        return np.nan
    month_idx = MONTH_ORDER.index(ts.month)
    if ts.day <= 15:
        bin_center = month_idx * 2
        bin_start, bin_end = 1, 15
    else:
        bin_center = month_idx * 2 + 1
        bin_start = 16
        bin_end = monthrange(ts.year, ts.month)[1]
    within = (ts.day - bin_start) / (bin_end - bin_start) if bin_end > bin_start else 0.5
    return bin_center - 0.5 + within


# Per-eddy-date scatter positioned by actual observation date
scatter_data = pig_medians.copy()
scatter_data["x"] = scatter_data["date"].apply(map_date_to_x)  # pyright: ignore[reportAttributeAccessIssue]
scatter_data = scatter_data.dropna(subset=["x"])  # pyright: ignore[reportCallIssue]
scatter_data = scatter_data[
    ~((scatter_data["polarity"] == "anticyclone") & (scatter_data["T chla"] > 1.5))
]

fig, axes = cast("tuple[Figure, np.ndarray]", plt.subplots(1, 2, figsize=(12, 5.5), sharey=True,
    gridspec_kw={"wspace": 0.08}))

summaries = {}
for pol in ["cyclone", "anticyclone"]:
    means, lo_all, hi_all, ns = [], [], [], []
    for m, h in bins_sequence:
        vals = eddy_hm.loc[
            (eddy_hm["polarity"] == pol)
            & (eddy_hm["month"] == m)
            & (eddy_hm["half"] == h),
            "T chla",
        ].to_numpy()
        if len(vals) == 0:
            means.append(np.nan); lo_all.append(np.nan); hi_all.append(np.nan); ns.append(0)
            continue
        if len(vals) >= 3:
            rng = np.random.default_rng(42)
            boots = np.array([
                np.mean(rng.choice(vals, size=len(vals), replace=True))
                for _ in range(2000)
            ])
            lo, hi = np.percentile(boots, [2.5, 97.5])
        else:
            lo, hi = np.min(vals), np.max(vals)
        means.append(np.mean(vals))
        lo_all.append(lo); hi_all.append(hi); ns.append(len(vals))
    summaries[pol] = (np.array(means), np.array(lo_all), np.array(hi_all), ns)

panel_info = [
    ("cyclone", "#2166ac"),
    ("anticyclone", "#b2182b"),
]

for ax, (pol, color) in zip(axes, panel_info):
    x_positions = np.arange(len(bins_sequence))

    # Scatter at actual date positions (one dot per per-eddy-date observation); low alpha so overlapping observations cumulatively darken dense regions
    sub = scatter_data[scatter_data["polarity"] == pol]
    ax.scatter(sub["x"], sub["T chla"], s=22, color=color, alpha=0.18,
        linewidths=0, zorder=1)

    means, lo_all, hi_all, ns = summaries[pol]
    valid = ~np.isnan(means)
    # lower row then upper row, the two-row layout errorbar expects: two (n_valid_bins,) -> (2, n_valid_bins)
    yerr = np.vstack([means[valid] - lo_all[valid], hi_all[valid] - means[valid]])
    ax.errorbar(x_positions[valid], means[valid], yerr=yerr, fmt="o-",
        color=color, lw=2.8, ms=8,
        capsize=5, capthick=1.8, elinewidth=1.8,
        markeredgecolor="white", markeredgewidth=0.8,
        zorder=3)

    ax.set_xticks(month_tick_positions)
    ax.set_xticklabels(MONTH_LABELS, fontsize=11)
    ax.set_xlabel("Month", fontsize=12, labelpad=8)
    ax.set_title(pol.capitalize(),
        fontsize=13, fontweight="bold", color=color, pad=12)
    ax.tick_params(axis="both", which="major", labelsize=10, length=4, width=1.0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(1.0)
    ax.grid(axis="y", alpha=0.22, linestyle="--", linewidth=0.6)
    ax.set_axisbelow(True)

y_max = max(eddy_hm["T chla"].max(), scatter_data["T chla"].max()) * 1.06
axes[0].set_ylim(0, y_max)
axes[0].set_ylabel("Mean T chl-a (mg m$^{-3}$)", fontsize=12, labelpad=10)

fig.suptitle("Mean Chl-a Over Time by Polarity",
    fontsize=15, y=1.00, fontweight="bold")
fig.tight_layout()
if SAVE_DIR is not None:
    fig.savefig(SAVE_DIR / "3_monthly_tchla_by_polarity.png", dpi=300, bbox_inches="tight")
plt.show()